# Campaign Performance Optimizer

## Objetivos de Aprendizaje

En este notebook, aprenderás a learn how to:

1. **Build multi-touch attribution** using SQL window functions
2. **Calculate channel ROI** across customer touchpoints
3. **Use `ROW_NUMBER()` and `COUNT() OVER()`** for journey sequencing
4. **Analyze conversion paths** with SQL analytics
5. **Optimize marketing spend** allocation based on data
6. **Create interactive dashboards** with Streamlit

---

## What You'll Build

A **marketing attribution system** that tracks customer journeys and calculates ROI:
- Multi-touch attribution (linear and time-decay models)
- Channel performance analysis (ROI, conversion rates)
- Campaign effectiveness metrics (cost per conversion)
- Budget optimization recommendations

---

## Technical Requirements

- **Snowflake account** with standard SQL access
- **Approximately 12 minutes** to complete
- **100% SQL** (except Streamlit dashboard)
- **No ML required** - Pure SQL analytics

---

## Key Snowflake Features Used

- `GENERATOR()` - Create synthetic campaign data
- `ROW_NUMBER() OVER()` - Sequence touchpoints in customer journey
- `COUNT() OVER()` - Calculate total touches per customer
- Window functions - Partition by customer for journey analysis
- `UUID_STRING()` - Generar unique identifiers
- Streamlit - Interactive campaign dashboard

Let's begin!

---

## Paso 1: Configuración del Entorno

### Qué Vamos a Hacer

Creating isolated Snowflake objects for this campaign analytics tutorial:
- **Database**: `CAMPAIGN_ANALYTICS_DB` - Container for all marketing data
- **Schema**: `PUBLIC` - Namespace within the database
- **Warehouse**: `COMPUTE_WH` - Compute resources for queries

### Why This Matters

Using dedicated objects keeps this tutorial isolated from your production environment and makes cleanup easy if needed.

### Multi-Touch Attribution

Unlike **first-touch** or **last-touch** attribution (which credit only one touchpoint), multi-touch attribution gives **proportional credit** to all touchpoints in the customer journey. This provides a more accurate view of channel effectiveness.

In [ ]:
CREATE DATABASE IF NOT EXISTS CAMPAIGN_ANALYTICS_DB;
USE SCHEMA CAMPAIGN_ANALYTICS_DB.PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL';
USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() as db;

---

## Paso 2: Create Data Tables

### Qué Vamos a Hacer

Creating three tables to track the customer journey:
1. **`campaigns`** - Campaign metadata (budget, channel, dates)
2. **`touchpoints`** - Every customer interaction with campaigns
3. **`conversions`** - Final conversion events (purchases, signups)

### Why This Structure?

Separating touchpoints from conversions lets us:
- Track **all interactions**, not just conversions
- Calculate **conversion rate** (conversions / touchpoints)
- Build **journey paths** by joining touchpoints to conversions

### Key Fields

- **`customer_id`**: Links touchpoints and conversions
- **`touchpoint_date`**: Timestamp for sequencing journey
- **`channel`**: Email, Social, Search, Display

In [ ]:
CREATE OR REPLACE TABLE campaigns (
    campaign_id VARCHAR(50) PRIMARY KEY,
    campaign_name VARCHAR(200),
    channel VARCHAR(50),
    start_date DATE,
    budget_usd DECIMAL(15,2)
);

CREATE OR REPLACE TABLE touchpoints (
    touchpoint_id VARCHAR(50) PRIMARY KEY,
    customer_id VARCHAR(50),
    campaign_id VARCHAR(50),
    touchpoint_date TIMESTAMP,
    channel VARCHAR(50),
    touchpoint_type VARCHAR(50)
);

CREATE OR REPLACE TABLE conversions (
    conversion_id VARCHAR(50) PRIMARY KEY,
    customer_id VARCHAR(50),
    conversion_date TIMESTAMP,
    conversion_value_usd DECIMAL(10,2)
);

---

## Paso 3: Generar Datos Sintéticos Campaign Data

### Qué Vamos a Hacer

Creating realistic marketing data:
- **20 campaigns** across 4 channels (Email, Social, Search, Display)
- **200 touchpoints** (customer interactions with campaigns)
- **30% conversion rate** (realistic for B2B marketing)

### The `GENERATOR()` Function

`GENERATOR(ROWCOUNT => N)` creates N rows instantly:
- Combined with `UNIFORM()` for random values
- `UUID_STRING()` creates unique IDs
- `DATEADD()` creates timestamps in the past 90 days

### Data Patterns

We're simulating realistic behavior:
- **Budgets**: \$50K - \$500K per campaign
- **Touchpoints**: Spread over past 90 days
- **Conversions**: 30% of customers convert after touchpoints
- **Conversion timing**: 1-72 hours after last touchpoint

This synthetic data is designed for demonstration - replace with your real campaign data in production.

In [ ]:
-- Insert campaigns
INSERT INTO campaigns
SELECT 
    'CAMP' || LPAD(SEQ4(), 4, '0') as campaign_id,
    channel || ' Campaign ' || SEQ4() as campaign_name,
    channel,
    DATEADD('day', -90, CURRENT_DATE()) as start_date,
    FLOOR(UNIFORM(50000, 500000, RANDOM())) as budget_usd
FROM (SELECT * FROM (VALUES ('Email'),('Social'),('Search'),('Display')) t(channel)),
TABLE(GENERATOR(ROWCOUNT => 5));

-- Generar touchpoints (customer journey)
INSERT INTO touchpoints
SELECT 
    UUID_STRING() as touchpoint_id,
    'CUST' || LPAD(FLOOR(UNIFORM(1, 1001, RANDOM())), 5, '0') as customer_id,
    'CAMP' || LPAD(FLOOR(UNIFORM(1, 21, RANDOM())), 4, '0') as campaign_id,
    DATEADD('hour', -FLOOR(UNIFORM(1, 2160, RANDOM())), CURRENT_TIMESTAMP()) as touchpoint_date,
    c.channel,
    CASE WHEN UNIFORM(0,1,RANDOM()) > 0.5 THEN 'Click' ELSE 'View' END as touchpoint_type
FROM campaigns c,
TABLE(GENERATOR(ROWCOUNT => 200));

-- Generar conversions (30% of customers convert)
INSERT INTO conversions
SELECT 
    UUID_STRING() as conversion_id,
    t.customer_id,
    DATEADD('hour', FLOOR(UNIFORM(1, 72, RANDOM())), MAX(t.touchpoint_date)) as conversion_date,
    ROUND(UNIFORM(1000, 5000, RANDOM()), 2) as conversion_value_usd
FROM touchpoints t
WHERE UNIFORM(0,1,RANDOM()) > 0.7
GROUP BY t.customer_id;

SELECT (SELECT COUNT(DISTINCT campaign_id) FROM campaigns) as campaigns,
       (SELECT COUNT(*) FROM touchpoints) as touchpoints,
       (SELECT COUNT(*) FROM conversions) as conversions;

---

## Paso 4: Multi-Touch Attribution Analysis

### Qué Vamos a Hacer

Calculating how much credit each touchpoint deserves for conversions using **two attribution models**:
1. **Linear Attribution**: Equal credit to all touchpoints
2. **Time-Decay Attribution**: More credit to recent touchpoints

### The `ROW_NUMBER()` Window Function

`ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY touchpoint_date)` assigns a sequence number to each touchpoint:
- **Partition by customer**: Each customer gets their own sequence (1, 2, 3...)
- **Order by date**: Earliest touchpoint = 1, latest = N
- **Result**: We can identify first touch, last touch, and position in journey

### Attribution Formula

**Linear Attribution**:
```sql
conversion_value / total_touches
```
Example: \$1,000 conversion with 5 touches = \$200 per touch

**Time-Decay Attribution**:
```sql
conversion_value × (touch_sequence / sum_of_all_sequences)
```
Example: Touch 4 of 5 gets more credit than touch 1 of 5

### Why Use Both Models?

- **Linear**: Good for understanding overall channel contribution
- **Time-Decay**: Good for understanding which channels close deals

In [ ]:
-- Calculate attribution
CREATE OR REPLACE TABLE attribution_analysis AS
WITH customer_journeys AS (
    SELECT 
        t.customer_id,
        t.touchpoint_id,
        t.campaign_id,
        t.channel,
        t.touchpoint_date,
        c.conversion_date,
        c.conversion_value_usd,
        ROW_NUMBER() OVER (PARTITION BY t.customer_id ORDER BY t.touchpoint_date) as touch_sequence,
        COUNT(*) OVER (PARTITION BY t.customer_id) as total_touches
    FROM touchpoints t
    LEFT JOIN conversions c ON t.customer_id = c.customer_id
    WHERE c.conversion_id IS NOT NULL
),
attribution_weights AS (
    SELECT 
        *,
        -- Linear attribution: equal credit to all touches
        conversion_value_usd / total_touches as linear_attribution,
        -- Time-decay: more recent touches get more credit
        conversion_value_usd * (touch_sequence::FLOAT / ((total_touches * (total_touches + 1)) / 2.0)) as time_decay_attribution
    FROM customer_journeys
)
SELECT 
    campaign_id,
    channel,
    COUNT(DISTINCT customer_id) as conversions_influenced,
    ROUND(SUM(linear_attribution), 2) as linear_attributed_revenue,
    ROUND(SUM(time_decay_attribution), 2) as time_decay_attributed_revenue
FROM attribution_weights
GROUP BY campaign_id, channel;

-- View attribution results
SELECT channel, 
       SUM(conversions_influenced) as total_conversions,
       ROUND(SUM(linear_attributed_revenue), 0) as total_revenue
FROM attribution_analysis
GROUP BY channel
ORDER BY total_revenue DESC;

---

## Paso 5: Calculate Channel ROI

### Qué Vamos a Hacer

Computing return on investment for each marketing channel:
- **Total spend**: Sum of campaign budgets per channel
- **Attributed revenue**: Sum of conversion values credited to channel
- **ROI**: (Revenue - Spend) / Spend × 100

### The `SUM() OVER()` Window Function

`SUM(time_decay_value) OVER (PARTITION BY channel)` calculates total attributed value per channel while keeping row-level detail:
- **Without `OVER()`**: `SUM()` would collapse rows (aggregate)
- **With `OVER()`**: `SUM()` adds a column with the total (window function)

This lets us see both individual touchpoints AND channel totals in one query.

### Interpreting ROI

- **ROI > 0%**: Channel is profitable (revenue exceeds spend)
- **ROI = 200%**: Every \$1 spent returns \$3 (\$1 + \$2 profit)
- **ROI < 0%**: Channel is losing money (needs optimization)

**Example**: 
- Spend: \$100,000
- Revenue: \$350,000
- ROI: (\$350K - \$100K) / \$100K = 250%

In [ ]:
-- ROI analysis by channel
CREATE OR REPLACE VIEW channel_roi AS
WITH spend_by_channel AS (
    SELECT channel, SUM(budget_usd) as total_spend
    FROM campaigns
    GROUP BY channel
),
revenue_by_channel AS (
    SELECT channel, SUM(linear_attributed_revenue) as total_revenue
    FROM attribution_analysis
    GROUP BY channel
)
SELECT 
    s.channel,
    s.total_spend,
    COALESCE(r.total_revenue, 0) as total_revenue,
    ROUND((total_revenue - total_spend), 0) as net_profit,
    ROUND((total_revenue / NULLIF(total_spend, 0)), 2) as roi_ratio,
    ROUND(((total_revenue - total_spend) / NULLIF(total_spend, 0)) * 100, 1) as roi_pct,
    CASE 
        WHEN roi_pct >= 200 THEN '🟢 Excellent'
        WHEN roi_pct >= 100 THEN '🟡 Good'
        WHEN roi_pct >= 0 THEN '🟠 Break-even'
        ELSE '🔴 Losing Money'
    END as roi_rating
FROM spend_by_channel s
LEFT JOIN revenue_by_channel r ON s.channel = r.channel
ORDER BY roi_pct DESC;

SELECT * FROM channel_roi;

---

## Paso 6: Interactive Campaign Dashboard

### Lo Que Verás

An interactive **Streamlit dashboard** showing:
- **Key metrics**: Total spend, revenue, conversions, ROI
- **Channel performance**: ROI comparison across channels
- **Attribution models**: Linear vs. time-decay comparison
- **Top campaigns**: Best performing campaigns by revenue

### How to Use

The dashboard is **embedded in this notebook** - no separate deployment needed!

Run this cell to launch the dashboard and explore your campaign performance interactively.

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title("📊 Campaign Performance Optimizer")

# ROI by channel
roi = session.sql("SELECT * FROM channel_roi ORDER BY roi_pct DESC").to_pandas()

col1, col2 = st.columns(2)
with col1:
    best_channel = roi.iloc[0]['CHANNEL'] if len(roi) > 0 else 'N/A'
    best_roi = roi.iloc[0]['ROI_PCT'] if len(roi) > 0 else 0
    st.metric("Best Channel", best_channel, f"+{best_roi:.1f}% ROI")
with col2:
    total_revenue = roi['TOTAL_REVENUE'].sum()
    total_spend = roi['TOTAL_SPEND'].sum()
    overall_roi = ((total_revenue - total_spend) / total_spend * 100) if total_spend > 0 else 0
    st.metric("Overall ROI", f"{overall_roi:.1f}%")

st.subheader("💰 Channel ROI Analysis")
st.dataframe(roi, use_container_width=True, hide_index=True)

# Attribution comparison
st.subheader("🎯 Attribution by Channel")
attribution = session.sql("""
    SELECT channel, 
           SUM(conversions_influenced) as conversions,
           ROUND(SUM(linear_attributed_revenue),0) as revenue
    FROM attribution_analysis
    GROUP BY channel
    ORDER BY revenue DESC
""").to_pandas()

st.bar_chart(attribution.set_index('CHANNEL')['REVENUE'])

# Campaign performance
st.subheader("📈 Top Performing Campaigns")
campaigns = session.sql("""
    SELECT c.campaign_name, c.channel, c.budget_usd,
           a.conversions_influenced, a.linear_attributed_revenue
    FROM campaigns c
    LEFT JOIN attribution_analysis a ON c.campaign_id = a.campaign_id
    ORDER BY a.linear_attributed_revenue DESC
    LIMIT 10
""").to_pandas()

st.dataframe(campaigns, use_container_width=True, hide_index=True)

---

## Paso 7: Analyze Results

### Understanding Attribution Results

The final view shows:
- **Total Spend**: Campaign budget invested per channel
- **Linear Revenue**: Revenue credited using equal attribution
- **Time-Decay Revenue**: Revenue credited with recency bias
- **ROI**: Return on investment percentage

### Key Insights

Look for:
1. **Alto ROI channels**: Invest more budget here
2. **Bajo ROI channels**: Optimize or reduce spend
3. **Linear vs. Time-Decay differences**: Shows which channels start vs. close journeys

**Example Interpretation**:
- **Email has high linear, lower time-decay**: Good for awareness, not closing
- **Search has high time-decay**: Customers search right before converting

---

##  Complete!

### What You Learned

1.  Multi-touch attribution with SQL window functions
2.  Calculate ROI by channel
3.  Analyze customer journey touchpoints
4.  Linear vs. time-decay attribution models
5.  Build marketing analytics dashboards

### Attribution Models

**Linear Attribution:**
```sql
conversion_value / total_touches
```
Equal credit to all touchpoints.

**Time-Decay Attribution:**
```sql
conversion_value * (touch_sequence / sum_of_sequences)
```
More recent touchpoints get more credit.

### Key Metrics

- **ROI Ratio**: Revenue / Spend
- **ROI %**: (Revenue - Spend) / Spend × 100
- **Attribution**: Revenue credit per channel

### Applications

- Budget allocation optimization
- Channel mix strategy
- Campaign performance tracking
- Marketing spend efficiency

[Snowflake Window Functions](https://docs.snowflake.com/en/sql-reference/functions-analytic)

---

## Limpieza del Entorno (Opcional)

### Qué Hace Esto

This cell will **completely remove** all objects created in this tutorial:
- Drops the `CAMPAIGN_ANALYTICS_DB` database (and all tables within it)
- Drops the `COMPUTE_WH` warehouse

### When to Use

Run this if you want to:
- Clean up your Snowflake environment after completing the tutorial
- Start fresh and re-run the entire notebook
- Remove all demo data

### Instructions

**This cell is commented out by default** to prevent accidental deletion when running "Run All".

To reset your environment:
1. **Remove the comment markers** (`--`) from the SQL commands below
2. **Run this cell manually**

 **Warning**: This action cannot be undone. All data will be permanently deleted.

In [ ]:
-- Descomentar las líneas siguientes and ejecutar esta celda para limpiar el entorno

-- DROP DATABASE IF EXISTS CAMPAIGN_ANALYTICS_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;
